# E001A — Public Replay Acquisition

This notebook fills the missing bridge between the official **Kaggriculture Episodes Index** and the replay parser.

## Input checklist
- **Input 1:** `kaggle/kaggriculture-episodes-index`
- **Input 2:** your `kaggriculture-code-repo` Dataset
- **Accelerator:** None / CPU
- **Internet:** **ON**
- **Competition rules:** you must have joined Kaggriculture / accepted its rules

## Outputs
- `/kaggle/working/kagv2/replays/episode-<id>-replay.json`
- `/kaggle/working/kagv2/replay_download_manifest.csv`
- `/kaggle/working/kagv2/selected_episode_ids.csv`

### Start small
The default downloads 250 episodes. Once the full pipeline is green, increase this to 1,000–5,000+ and use stronger stratified selection.


In [ ]:
from pathlib import Path
import sys, os, json, subprocess, time, random
import pandas as pd
import numpy as np

ROOTS=[Path('/kaggle/input'),Path('/kaggle/working'),Path.cwd()]
repo_file=next((p for r in ROOTS if r.exists() for p in r.rglob('src/kagv2/schema.py')),None)
if repo_file is None:
    raise FileNotFoundError('Attach your kaggriculture-code-repo Dataset.')
ROOT=repo_file.parents[2]
sys.path.insert(0,str(ROOT))
WORK=Path('/kaggle/working/kagv2')
WORK.mkdir(parents=True,exist_ok=True)
REPLAY_OUT=WORK/'replays'
REPLAY_OUT.mkdir(parents=True,exist_ok=True)
print('ROOT=',ROOT)
print('WORK=',WORK)


In [ ]:
from src.kagv2.schema import choose_index_table, normalize_index

EP_ROOT=Path('/kaggle/input/kaggriculture-episodes-index')
if not EP_ROOT.exists():
    hits=[p for p in Path('/kaggle/input').glob('*') if 'kaggriculture' in p.name.lower() and 'episode' in p.name.lower()]
    if not hits:
        raise FileNotFoundError('Attach kaggle/kaggriculture-episodes-index')
    EP_ROOT=hits[0]

p, raw = choose_index_table(EP_ROOT)
catalog = normalize_index(raw)
print('index file:', p)
print('catalog shape:', catalog.shape)
print('columns:', list(catalog.columns))
display(catalog.head())

if 'episode_id' not in catalog:
    raise RuntimeError('Could not infer episode_id. Run E000 and send episode_schema_report.csv so we can add a schema adapter.')


In [ ]:
# Current-engine filtering and replay selection.
MAX_REPLAYS = 250
SEED = 20260817
rng = np.random.default_rng(SEED)

d = catalog.copy()

if 'created_at' in d.columns:
    d = d[d['created_at'].notna()].copy()
    post = d[d['created_at'] >= pd.Timestamp('2026-08-07', tz='UTC')]
    if len(post):
        d = post
        print('Filtered to post-2026-08-07 rows:', len(d))

# Make episode IDs clean strings.
def clean_episode_id(x):
    if pd.isna(x):
        return None
    try:
        fx=float(x)
        if fx.is_integer():
            return str(int(fx))
    except Exception:
        pass
    return str(x).strip()

d['episode_id_clean'] = d['episode_id'].map(clean_episode_id)
d = d[d['episode_id_clean'].notna() & (d['episode_id_clean']!='')].copy()

# Collapse one-row-per-agent indexes to one row per episode.
agg = {'episode_id_clean':'first'}
for c in ['created_at','rating','reward','submission_id','team_name','team_id']:
    if c in d.columns:
        agg[c] = 'max' if c in ['rating','reward','created_at'] else 'first'
eps = d.groupby('episode_id_clean', as_index=False).agg(agg)
eps = eps.rename(columns={'episode_id_clean':'episode_id'})
print('unique candidate episodes:', len(eps))

# Stratified choice:
# 50% strongest if a rating exists, 30% most recent, 20% random diversity.
chosen=[]
n=MAX_REPLAYS
if 'rating' in eps.columns and pd.to_numeric(eps['rating'], errors='coerce').notna().sum() >= 20:
    eps['_rating']=pd.to_numeric(eps['rating'], errors='coerce')
    chosen += eps.sort_values('_rating',ascending=False).head(max(1,n//2))['episode_id'].tolist()

if 'created_at' in eps.columns and eps['created_at'].notna().sum() >= 20:
    chosen += eps.sort_values('created_at',ascending=False).head(max(1,int(n*.30)))['episode_id'].tolist()

remaining=[x for x in eps['episode_id'].tolist() if x not in set(chosen)]
if remaining:
    k=min(len(remaining), max(0,n-len(set(chosen))))
    if k:
        chosen += rng.choice(remaining,size=k,replace=False).tolist()

# If rating/date weren't available, random sample.
chosen=list(dict.fromkeys(chosen))
if len(chosen) < min(n,len(eps)):
    remaining=[x for x in eps['episode_id'].tolist() if x not in set(chosen)]
    k=min(len(remaining), min(n,len(eps))-len(chosen))
    if k:
        chosen += rng.choice(remaining,size=k,replace=False).tolist()
chosen=chosen[:n]

selected=eps[eps['episode_id'].isin(chosen)].copy()
selected.to_csv(WORK/'selected_episode_ids.csv',index=False)
print('selected episodes:',len(chosen))
display(selected.head(20))


In [ ]:
# Verify Kaggle CLI and authentication before starting a large download.
print(subprocess.run(['kaggle','--version'],capture_output=True,text=True).stdout.strip())

probe = subprocess.run(
    ['kaggle','competitions','leaderboard','kaggriculture','-s','-v'],
    capture_output=True,text=True,timeout=60
)
print('leaderboard return code:', probe.returncode)
print(probe.stdout[:1500])
if probe.returncode != 0:
    print(probe.stderr[:2000])
    raise RuntimeError(
        'Kaggle CLI could not access the competition. Confirm Internet=ON and that you joined/accepted the Kaggriculture rules.'
    )


In [ ]:
# Download replays. Sequential by default to be gentle with the API.
# Existing files are skipped, so this notebook is restart-safe.
rows=[]
for i,eid in enumerate(chosen,1):
    expected=REPLAY_OUT/f'episode-{eid}-replay.json'
    if expected.exists() and expected.stat().st_size > 100:
        rows.append({'episode_id':eid,'status':'cached','bytes':expected.stat().st_size,'error':''})
        continue

    cmd=['kaggle','competitions','replay',str(eid),'-p',str(REPLAY_OUT)]
    ok=False
    err=''
    for attempt in range(3):
        try:
            r=subprocess.run(cmd,capture_output=True,text=True,timeout=120)
            if r.returncode==0:
                # The CLI's canonical filename is episode-<id>-replay.json.
                candidates=list(REPLAY_OUT.glob(f'*{eid}*replay*.json'))
                if candidates:
                    expected=candidates[0]
                if expected.exists() and expected.stat().st_size>100:
                    ok=True
                    break
            err=(r.stderr or r.stdout)[-1500:]
        except Exception as ex:
            err=repr(ex)
        time.sleep(1.5*(attempt+1))

    rows.append({
        'episode_id':eid,
        'status':'ok' if ok else 'failed',
        'bytes':expected.stat().st_size if expected.exists() else 0,
        'error':err
    })

    if i % 25 == 0 or not ok:
        man=pd.DataFrame(rows)
        man.to_csv(WORK/'replay_download_manifest.csv',index=False)
        print(f'{i}/{len(chosen)}  ok={(man.status.isin(["ok","cached"])).sum()}  failed={(man.status=="failed").sum()}')

manifest=pd.DataFrame(rows)
manifest.to_csv(WORK/'replay_download_manifest.csv',index=False)
display(manifest.status.value_counts())
display(manifest[manifest.status=='failed'].head(20))
print('usable replay files:', len(list(REPLAY_OUT.glob('*replay*.json'))))


In [ ]:
# Hard gate: E001B should only run when acquisition succeeded.
files=sorted(REPLAY_OUT.glob('*replay*.json'))
if not files:
    raise RuntimeError('No replay downloads succeeded. Inspect replay_download_manifest.csv before continuing.')

print('READY FOR E001B')
print('replays:',len(files))
print('total MB:',round(sum(p.stat().st_size for p in files)/1024**2,2))
print('first files:')
for p in files[:5]:
    print(' ',p.name,p.stat().st_size)


## Next step

Save this notebook's output as a Kaggle Dataset, or keep the same session alive, then run **E001B — Replay Data Factory** with Internet OFF.

For serious training, increase `MAX_REPLAYS` after the 250-episode smoke run succeeds. We will later refine selection using the actual E000 schema and Bradley–Terry/top-ladder metadata.
